# Exploratory Data Analysis (EDA) with pandas and polars

In this notebook we explore the Olist e-commerce CSV files **twice**: once with **pandas** and once with **polars**. Every step gives the same answer in both libraries, so you can compare how they work and decide which you prefer.

### What you will learn
- What EDA is and why we do it **before** we design or build anything
- How to load CSV files and choose the right data types
- How to look at the shape, types and a sample of a table
- How to find **missing values** and **duplicates**
- How to describe **categorical** and **numeric** columns, and spot outliers
- How to look at data **over time**
- How to **join** tables safely and check the result
- How to measure simple **relationships** between columns
- The main differences between pandas and polars

### What is EDA?
**Exploratory Data Analysis** means looking at a dataset *before* using it, to learn what is in it and what is wrong with it. We are not answering a business question yet. We are asking questions about the **data itself**:

| Question | Why it matters |
|---|---|
| How big is it? What columns and types? | Know what you have |
| What is missing? What is duplicated? | Data quality problems break later steps |
| What do the values look like? Any outliers? | Averages can be misleading |
| Does it cover the time period we expect? | Incomplete months distort trends |
| How do the tables connect? | Wrong joins double count |

What we learn here feeds directly into design decisions later. For example, the missing product categories we find here are the reason the star schema notebook replaces them with `'unknown'`.

> **How to work with this notebook:** run the cells from top to bottom. For each step, read the explanation first, run the **pandas** cell, then the **polars** cell, and compare the output and the code.

### pandas versus polars in one minute

| | pandas | polars |
|---|---|---|
| Style | Methods on a DataFrame, square brackets: `df[df["price"] > 100]` | **Expressions**: `df.filter(pl.col("price") > 100)` |
| Row index | Yes (every row has an index label) | **No** index |
| Missing values | `NaN` / `None`; integer columns with missing values become **float** | `null` for every type; integers stay integers |
| Execution | Runs each step immediately (eager) | Eager **and** lazy (plans the whole query first) |
| Plotting | Built in: `df.plot()` | None built in: use matplotlib |
| Where you meet it | Everywhere: tutorials, libraries, jobs | Newer, very fast, growing quickly |

You do not need to pick a winner. Many people use both.

## 1. Setup

### 1.1 Imports

- `pathlib.Path`: file paths that work on any operating system
- `pandas as pd` and `polars as pl`: the two libraries. The short names `pd` and `pl` are the standard convention
- `numpy as np`: numbers and arrays (used for logarithms)
- `matplotlib.pyplot as plt`: charts. Both libraries can feed data to matplotlib

In [18]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

print("pandas", pd.__version__, "| polars", pl.__version__)

pandas 3.0.6 | polars 2.1.0


### 1.2 Find the CSV files

The Olist data comes as several CSV files. The function below searches the `data` folder for a file whose name contains a keyword (for example `orders`) and stops with a clear message if it cannot find it. We store the paths in a dictionary called `CSV`.

`Path.cwd().parent` is the folder above the one this notebook is in, which is the project root.

In [2]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"


def find_csv(keyword: str) -> Path:
    """Return the first CSV file under data/ whose name contains the keyword."""
    matches = sorted(DATA_DIR.rglob(f"*{keyword}*.csv"))
    assert matches, f"No CSV with '{keyword}' in its name found under {DATA_DIR}. Put the Olist CSV files in the data folder."
    return matches[0]


CSV = {name: find_csv(name) for name in ["customers", "orders", "order_items", "products", "sellers"]}
pd.Series({name: str(path.relative_to(ROOT)) for name, path in CSV.items()}, name="file").to_frame()

,file
customers,data\raw\olist_customers_dataset.csv
orders,data\raw\olist_orders_dataset.csv
order_items,data\raw\olist_order_items_dataset.csv
products,data\raw\olist_products_dataset.csv
sellers,data\raw\olist_sellers_dataset.csv


## 2. Load the CSV files

A CSV file is plain text: it has **no data types**. The library has to guess whether a column is a number, a date or text. Guessing is often right, but not always, so we help in two places:

1. **Dates.** Without help, dates stay as text and we cannot calculate with them. We tell pandas which columns are dates with `parse_dates=[...]`. In polars we use `try_parse_dates=True`.
2. **Zip codes.** A zip code such as `01046` looks like a number, but it is an **identifier**: we never add or average zip codes, and the leading zero matters. If it is read as a number it becomes `1046`. We force it to text: `dtype=...` in pandas and `schema_overrides=...` in polars.

> **Rule of thumb:** a column is only a number if it makes sense to do arithmetic with it.

### 2.1 Load with pandas

In [12]:
ORDER_DATES = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders_pd    = pd.read_csv(CSV["orders"], parse_dates=ORDER_DATES)
items_pd     = pd.read_csv(CSV["order_items"], parse_dates=["shipping_limit_date"])
products_pd  = pd.read_csv(CSV["products"])
customers_pd = pd.read_csv(CSV["customers"], dtype={"customer_zip_code_prefix": "str"})
sellers_pd   = pd.read_csv(CSV["sellers"], dtype={"seller_zip_code_prefix": "str"})

tables_pd = {"orders": orders_pd, "order_items": items_pd, "products": products_pd,
             "customers": customers_pd, "sellers": sellers_pd}

### 2.2 Load with polars

In [4]:
orders_pl    = pl.read_csv(CSV["orders"], try_parse_dates=True)
items_pl     = pl.read_csv(CSV["order_items"], try_parse_dates=True)
products_pl  = pl.read_csv(CSV["products"])
customers_pl = pl.read_csv(CSV["customers"], schema_overrides={"customer_zip_code_prefix": pl.String})
sellers_pl   = pl.read_csv(CSV["sellers"], schema_overrides={"seller_zip_code_prefix": pl.String})

tables_pl = {"orders": orders_pl, "order_items": items_pl, "products": products_pl,
             "customers": customers_pl, "sellers": sellers_pl}

## 3. A first look at the data

Always start with the same four questions: **how big** is each table, **what do the first rows** look like, **what types** do the columns have, and **what are typical values**?

### 3.1 How big is each table?

`shape` is a (rows, columns) pair. We collect it for all five tables.

In [20]:
# pandas
pd.DataFrame(
    [(name, df.shape[0], df.shape[1]) for name, df in tables_pd.items()],
    columns=["table", "rows", "columns"],
)

,table,rows,columns
0,orders,99441,8
1,order_items,112650,7
2,products,32951,9
3,customers,99441,5
4,sellers,3095,4


In [21]:
# polars
pl.DataFrame(
    [(name, df.shape[0], df.shape[1]) for name, df in tables_pl.items()],
    schema=["table", "rows", "columns"],
    orient="row",
)

table,rows,columns
str,i64,i64
"""orders""",99441,8
"""order_items""",112650,7
"""products""",32951,9
"""customers""",99441,5
"""sellers""",3095,4


### 3.2 What do the first rows look like?

`head(n)` shows the first `n` rows. Look at the actual values: do they match the column names? Are the formats what you expect?

Notice one visible difference: pandas shows a **row index** (0, 1, 2...) on the left, polars does not have one and shows the **column types** under each column name.

In [22]:
# pandas
orders_pd.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04


In [23]:
# polars
orders_pl.head(3)

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
str,str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs]
"""e481f51cbdc54678b7cc49136f2d6a…","""9ef432eb6251297304e76186b10a92…","""delivered""",2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
"""53cdb2fc8bc7dce0b6741e21502734…","""b0830fb4747a6c6d20dea0b8c802d7…","""delivered""",2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
"""47770eb9100c2d0c44946d9cf07ec6…","""41ce2a54c0b03bf3443c3d931a3670…","""delivered""",2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


### 3.3 What are the column types?

Check that every column has the type you expect. Dates should be dates, identifiers should be text, amounts should be numbers.

- pandas: `.dtypes` (or `.info()`, which also shows non-null counts)
- polars: `.schema`

In pandas, text columns show as `object` or `str` depending on the version, and dates as `datetime64`. In polars they are `String` and `Datetime`.

In [25]:
# pandas
orders_pd.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 13.0 MB


In [24]:
# polars
orders_pl.schema

Schema([('order_id', String),
        ('customer_id', String),
        ('order_status', String),
        ('order_purchase_timestamp', Datetime(time_unit='us', time_zone=None)),
        ('order_approved_at', Datetime(time_unit='us', time_zone=None)),
        ('order_delivered_carrier_date',
         Datetime(time_unit='us', time_zone=None)),
        ('order_delivered_customer_date',
         Datetime(time_unit='us', time_zone=None)),
        ('order_estimated_delivery_date',
         Datetime(time_unit='us', time_zone=None))])

### 3.4 What are typical values? (`describe`)

`describe()` gives summary statistics for each column: count, mean, standard deviation, minimum, quartiles and maximum. Look at `order_items`: the **mean** price is well above the **median** (50%), and the maximum is far away from the rest. We will come back to this in section 6.

In [ ]:
# pandas
items_pd[["price", "freight_value"]].describe()

In [ ]:
# polars
items_pl.select(["price", "freight_value"]).describe()

## 4. Data quality: missing values and duplicates

### 4.1 Missing values

Missing values are the most common data quality problem. We count them per column and also show them as a percentage of all rows. Only columns with at least one missing value are shown.

- pandas: `isna().sum()` counts missing values per column
- polars: `null_count()` returns one row with the count per column; `transpose` turns it into a tall table

In [7]:
# pandas
missing_pd = orders_pd.isna().sum().to_frame("missing")
missing_pd["pct"] = (100 * missing_pd["missing"] / len(orders_pd)).round(2)
missing_pd[missing_pd["missing"] > 0]

,missing,pct
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98


In [8]:
# polars
missing_pl = (
    orders_pl.null_count()
    .transpose(include_header=True, header_name="column", column_names=["missing"])
    .with_columns((100 * pl.col("missing") / orders_pl.height).round(2).alias("pct"))
    .filter(pl.col("missing") > 0)
)
missing_pl

column,missing,pct
str,u32,f64
"""order_approved_at""",160,0.16
"""order_delivered_carrier_date""",1783,1.79
"""order_delivered_customer_date""",2965,2.98


### 4.2 Is the missing data a problem, or does it make sense?

Not every missing value is an error. An order that has not been delivered yet **cannot** have a delivery date. So we check *where* the missing delivery dates are, by counting them per order status.

- pandas: `groupby("order_status")` splits the table into groups, then `agg` calculates per group
- polars: `group_by("order_status").agg(...)`, with `pl.len()` for the number of rows in each group

If the missing dates are almost all in statuses like `shipped` or `canceled`, they are expected. A `delivered` order with no delivery date would be a real problem.

In [9]:
# pandas
(
    orders_pd.assign(no_delivery_date=orders_pd["order_delivered_customer_date"].isna())
    .groupby("order_status")["no_delivery_date"]
    .agg(missing="sum", orders="count")
    .sort_values("orders", ascending=False)
)

,missing,orders
order_status,,
delivered,8,96478
shipped,1107,1107
canceled,619,625
unavailable,609,609
invoiced,314,314
processing,301,301
created,5,5
approved,2,2


In [ ]:
# polars
(
    orders_pl.group_by("order_status")
    .agg(
        pl.col("order_delivered_customer_date").is_null().sum().alias("missing"),
        pl.len().alias("orders"),
    )
    .sort("orders", descending=True)
)

### 4.3 A type difference caused by missing values

Remember the table at the top: in pandas, an integer column with missing values becomes **float**, because `NaN` is a float. Polars keeps it an integer and uses `null`. Compare the type of the same column in both libraries:

In [10]:
print("pandas:", products_pd["product_name_lenght"].dtype)
print("polars:", products_pl["product_name_lenght"].dtype)

pandas: float64
polars: Int64


### 4.4 Duplicates and keys

Every table should have a **key**: a column (or a combination of columns) that identifies a row uniquely. We check two things:

1. Are there rows that are **completely identical**?
2. Is the key really unique? (`order_id` in `orders`; the pair `order_id` + `order_item_id` in `order_items`.)

We expect **0** duplicates everywhere. If a key is not unique, joins later will multiply rows.

In [ ]:
# pandas
print("identical rows in orders:        ", orders_pd.duplicated().sum())
print("duplicate order_id in orders:    ", orders_pd["order_id"].duplicated().sum())
print("duplicate (order_id, order_item_id) in order_items:",
      items_pd.duplicated(subset=["order_id", "order_item_id"]).sum())

In [ ]:
# polars
print("identical rows in orders:        ", orders_pl.is_duplicated().sum())
print("duplicate order_id in orders:    ", orders_pl["order_id"].is_duplicated().sum())
print("duplicate (order_id, order_item_id) in order_items:",
      items_pl.select(["order_id", "order_item_id"]).is_duplicated().sum())

## 5. Categorical columns

For a column with a limited set of values (a status, a category, a state), the basic question is: **how often does each value occur?**

### 5.1 Order status

- pandas: `value_counts()` counts each value and sorts from most to least frequent
- polars: `group_by(...).agg(pl.len())`. This is the same pattern as in 4.2, so there is only one idea to learn

In [17]:
# pandas
status_pd = orders_pd["order_status"].value_counts().to_frame("orders")
status_pd["pct"] = (100 * status_pd["orders"] / status_pd["orders"].sum()).round(2)
status_pd

,orders,pct
order_status,,
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


In [ ]:
# polars
status_pl = (
    orders_pl.group_by("order_status")
    .agg(pl.len().alias("orders"))
    .with_columns((100 * pl.col("orders") / pl.col("orders").sum()).round(2).alias("pct"))
    .sort("orders", descending=True)
)
status_pl

Almost all orders are `delivered`. The other statuses are rare, so on a normal bar chart they would be invisible. That is a first hint that we may want to treat them separately later.

### 5.2 Product categories (and missing categories)

We replace missing categories with `'unknown'` so that they show up in the counts instead of silently disappearing. (`value_counts()` in pandas ignores missing values by default!) Then we take the 10 most frequent categories.

- pandas: `fillna("unknown")`
- polars: `fill_null("unknown")`

In [ ]:
# pandas
top_categories_pd = products_pd["product_category_name"].fillna("unknown").value_counts().head(10)
top_categories_pd

In [ ]:
# polars
top_categories_pl = products_pl["product_category_name"].fill_null("unknown").value_counts(sort=True).head(10)
top_categories_pl

### 5.3 A quick chart

Charts make categories easier to compare. pandas has plotting built in (`.plot`). Polars has no plotting of its own, but matplotlib accepts lists and arrays, so we pass `.to_list()`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

# pandas: built-in plotting
top_categories_pd.sort_values().plot.barh(ax=axes[0], color="#2a7de1")
axes[0].set_title("Top 10 categories (pandas)")
axes[0].set_ylabel("")

# polars: pass lists to matplotlib
cats = top_categories_pl.sort("count")
axes[1].barh(cats["product_category_name"].to_list(), cats["count"].to_list(), color="#e07a1f")
axes[1].set_title("Top 10 categories (polars)")

plt.tight_layout()
plt.show()

## 6. Numeric columns: distributions and outliers

For a numeric column such as `price` we want to know the **shape of the distribution**, not just the average.

### 6.1 Quantiles and skew

A **quantile** is a cut point: the 50% quantile (median) splits the rows in half; the 99% quantile means that 99% of the values are below it. **Skewness** measures how lopsided a distribution is: a large positive value means a long tail of very high values.

Compare the **mean** with the **median**. If the mean is much higher, a few very large values are pulling it up.

> **A detail to know:** when a quantile falls *between* two values, pandas **interpolates** (a weighted average of the two, called `linear`). Polars by default picks the **nearest** value. That can give slightly different numbers. We write `interpolation="linear"` in polars so that both libraries match.

In [ ]:
# pandas
price_pd = items_pd["price"]
pd.Series({
    "mean": price_pd.mean(),
    "median": price_pd.median(),
    "p90": price_pd.quantile(0.90),
    "p99": price_pd.quantile(0.99),
    "max": price_pd.max(),
    "skew": price_pd.skew(),
}).round(2).to_frame("price")

In [ ]:
# polars
items_pl.select(
    pl.col("price").mean().alias("mean"),
    pl.col("price").median().alias("median"),
    pl.col("price").quantile(0.90, interpolation="linear").alias("p90"),
    pl.col("price").quantile(0.99, interpolation="linear").alias("p99"),
    pl.col("price").max().alias("max"),
    pl.col("price").skew().alias("skew"),
).with_columns(pl.all().round(2))

### 6.2 Histogram

A histogram splits the values into bins and counts how many fall into each. Prices are **right-skewed**: most items are cheap and a few are very expensive, so the left chart is squeezed against the left edge. On a **log scale** (right chart) the shape becomes visible. Taking the logarithm is a standard trick for skewed money values.

Both libraries can feed matplotlib: pandas passes a Series directly, polars passes a NumPy array with `.to_numpy()`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

# pandas: raw price
axes[0].hist(items_pd["price"], bins=50, color="#2a7de1")
axes[0].set_title("Price (pandas, raw)")

# polars: log10 of price
axes[1].hist(np.log10(items_pl["price"].to_numpy()), bins=50, color="#e07a1f")
axes[1].set_title("log10(price) (polars)")

plt.tight_layout()
plt.show()

### 6.3 Outliers

An **outlier** is a value far from the others. A common rule (the IQR rule) calls a value an outlier if it is more than 1.5 × IQR above the third quartile, where **IQR = Q3 − Q1**.

> **Outliers are not automatically errors.** A very expensive item may be real. First look at them, then decide. Here we count them and show the 5 most expensive items.

In [ ]:
# pandas
q1, q3 = items_pd["price"].quantile([0.25, 0.75])
upper_pd = q3 + 1.5 * (q3 - q1)
print(f"upper limit: {upper_pd:.2f}  |  outliers: {(items_pd['price'] > upper_pd).sum()}")
items_pd.nlargest(5, "price")[["order_id", "product_id", "price", "freight_value"]]

In [ ]:
# polars
q1, q3 = items_pl.select(
    pl.col("price").quantile(0.25, interpolation="linear").alias("q1"),
    pl.col("price").quantile(0.75, interpolation="linear").alias("q3"),
).row(0)
upper_pl = q3 + 1.5 * (q3 - q1)
print(f"upper limit: {upper_pl:.2f}  |  outliers: {items_pl.filter(pl.col('price') > upper_pl).height}")
items_pl.top_k(5, by="price").select(["order_id", "product_id", "price", "freight_value"])

### 6.4 Impossible values

Some values are not just unusual but **impossible**: a price of zero or below, a negative freight cost. We check for them explicitly.

In [ ]:
# pandas
print("price <= 0:   ", (items_pd["price"] <= 0).sum())
print("freight < 0:  ", (items_pd["freight_value"] < 0).sum())

In [ ]:
# polars
print("price <= 0:   ", items_pl.filter(pl.col("price") <= 0).height)
print("freight < 0:  ", items_pl.filter(pl.col("freight_value") < 0).height)

## 7. Looking at data over time

### 7.1 Orders per month

Trends only make sense if the data covers the whole period. We count orders per month: we format each purchase timestamp as `YYYY-MM` and count the rows per month.

- pandas: `.dt.strftime("%Y-%m")` on a date column
- polars: the same method name, `.dt.strftime("%Y-%m")`, used inside an expression

At the end we compare the two results to prove they are identical.

In [ ]:
# pandas
monthly_pd = (
    orders_pd.groupby(orders_pd["order_purchase_timestamp"].dt.strftime("%Y-%m"))
    .size()
    .rename("orders")
    .rename_axis("month")
    .reset_index()
)
monthly_pd.head()

In [ ]:
# polars
monthly_pl = (
    orders_pl.group_by(pl.col("order_purchase_timestamp").dt.strftime("%Y-%m").alias("month"))
    .agg(pl.len().alias("orders"))
    .sort("month")
)
print("same result in both libraries:", monthly_pd["orders"].tolist() == monthly_pl["orders"].to_list())
monthly_pl.head()

In [ ]:
ax = monthly_pd.plot(x="month", y="orders", figsize=(12, 3.5), legend=False, marker="o")
ax.set_title("Orders per month")
ax.set_xlabel("")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

**Look at the first and last months.** They are often much lower than the rest, because the data collection started or stopped in the middle of that month. Comparing an incomplete month with a full one would give a wrong "drop" in sales. In your analysis you would usually restrict yourself to the months with full coverage.

### 7.2 Delivery time and late deliveries

Dates also let us create **new columns**:

- `delivery_days`: days between purchase and delivery
- `is_late`: delivered **after** the estimated date (we compare the dates only, so delivery at 15:00 on the estimated day is on time)

We only use **delivered** orders that have a delivery date. This is why we looked at missing values first.

- pandas: `assign(...)` adds columns; subtracting two date columns gives a time difference, and `.dt.days` turns it into a number
- polars: `with_columns(...)` adds columns; `.dt.total_days()` does the same

In [ ]:
# pandas
delivered_pd = (
    orders_pd.dropna(subset=["order_delivered_customer_date"])
    .assign(
        delivery_days=lambda d: (d["order_delivered_customer_date"] - d["order_purchase_timestamp"]).dt.days,
        is_late=lambda d: d["order_delivered_customer_date"].dt.normalize() > d["order_estimated_delivery_date"].dt.normalize(),
    )
)
print(f"late deliveries: {100 * delivered_pd['is_late'].mean():.1f}%")
delivered_pd["delivery_days"].describe().round(1).to_frame()

In [ ]:
# polars
delivered_pl = (
    orders_pl.filter(pl.col("order_delivered_customer_date").is_not_null())
    .with_columns(
        (pl.col("order_delivered_customer_date") - pl.col("order_purchase_timestamp")).dt.total_days().alias("delivery_days"),
        (pl.col("order_delivered_customer_date").dt.date() > pl.col("order_estimated_delivery_date").dt.date()).alias("is_late"),
    )
)
print(f"late deliveries: {100 * delivered_pl['is_late'].mean():.1f}%")
delivered_pl.select("delivery_days").describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(delivered_pd["delivery_days"].clip(upper=60), bins=60, color="#2a7de1")
ax.set_title("Delivery time in days (values above 60 grouped at 60)")
plt.tight_layout()
plt.show()

## 8. Combining tables: joins

The interesting questions need several tables. A **join** combines rows from two tables that share a key. Joins are also where most mistakes happen, so after every join we check **the number of rows**.

- A **left join** keeps every row of the left table, even without a match (the new columns are missing)
- If the right table has the key **twice**, the left rows get **duplicated**. This is the most common join bug

### 8.1 Before joining: do all products exist?

Every item should point to a product that exists. An item without a matching product is an **orphan**. An *anti join* returns the rows of the left table that have **no match**; we expect 0.

- pandas: `~df["col"].isin(other["col"])`
- polars: `join(..., how="anti")`

In [ ]:
# pandas
print("items without a product:", (~items_pd["product_id"].isin(products_pd["product_id"])).sum())
print("product_id unique in products:", products_pd["product_id"].is_unique)

In [ ]:
# polars
print("items without a product:", items_pl.join(products_pl, on="product_id", how="anti").height)
print("product_id unique in products:", products_pl["product_id"].is_unique().all())

### 8.2 Join items with products and measure revenue per category

We attach the product category to every order line, replace missing categories with `'unknown'`, and add up the price per category.

- pandas: `merge(..., on=..., how="left")`, then `groupby(...).agg(...)`
- polars: `join(..., on=..., how="left")`, then `group_by(...).agg(...)`

Right after the join we check that the number of rows did **not** change.

In [ ]:
# pandas
lines_pd = items_pd.merge(products_pd[["product_id", "product_category_name"]], on="product_id", how="left")
print("rows before / after join:", len(items_pd), "/", len(lines_pd))

revenue_pd = (
    lines_pd.assign(category=lines_pd["product_category_name"].fillna("unknown"))
    .groupby("category")["price"]
    .agg(items="count", revenue="sum", avg_price="mean")
    .sort_values("revenue", ascending=False)
    .round(2)
    .head(10)
)
revenue_pd

In [ ]:
# polars
lines_pl = items_pl.join(products_pl.select(["product_id", "product_category_name"]), on="product_id", how="left")
print("rows before / after join:", items_pl.height, "/", lines_pl.height)

revenue_pl = (
    lines_pl.with_columns(pl.col("product_category_name").fill_null("unknown").alias("category"))
    .group_by("category")
    .agg(
        pl.len().alias("items"),
        pl.col("price").sum().alias("revenue"),
        pl.col("price").mean().alias("avg_price"),
    )
    .sort("revenue", descending=True)
    .with_columns(pl.col("revenue", "avg_price").round(2))
    .head(10)
)
revenue_pl

In [ ]:
ax = revenue_pd["revenue"].sort_values().plot.barh(figsize=(8, 3.5), color="#2a7de1")
ax.set_title("Revenue by product category (top 10)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### 8.3 Join orders with customers: orders per state

Each order has a `customer_id`, and the `customers` table tells us where that customer lives. This is a **one-to-one** join (one customer record per order), so the row count must stay the same.

In [ ]:
# pandas
orders_state_pd = orders_pd.merge(customers_pd[["customer_id", "customer_state"]], on="customer_id", how="left")
print("rows before / after join:", len(orders_pd), "/", len(orders_state_pd))
orders_state_pd["customer_state"].value_counts().head(5).to_frame("orders")

In [ ]:
# polars
orders_state_pl = orders_pl.join(customers_pl.select(["customer_id", "customer_state"]), on="customer_id", how="left")
print("rows before / after join:", orders_pl.height, "/", orders_state_pl.height)
orders_state_pl.group_by("customer_state").agg(pl.len().alias("orders")).sort("orders", descending=True).head(5)

## 9. Relationships between numeric columns

Do expensive items also have higher freight cost? The **correlation coefficient** measures how strongly two columns move together: `+1` is a perfect positive relationship, `0` no linear relationship, `-1` a perfect negative one.

Two kinds are common:
- **Pearson**: measures *linear* relationships; sensitive to outliers
- **Spearman**: works on the *ranks* of the values; more robust for skewed data like prices

> **Correlation is not causation.** Two columns can move together without one causing the other.

In [ ]:
# pandas
print("Pearson: ", round(items_pd["price"].corr(items_pd["freight_value"]), 3))
print("Spearman:", round(items_pd["price"].corr(items_pd["freight_value"], method="spearman"), 3))

In [ ]:
# polars
print("Pearson: ", round(items_pl.select(pl.corr("price", "freight_value")).item(), 3))
print("Spearman:", round(items_pl.select(pl.corr("price", "freight_value", method="spearman")).item(), 3))

A scatter plot shows what a single number hides. With many thousands of points we draw a **random sample** so the chart stays readable (`sample` exists in both libraries; the seed makes it repeatable).

In [ ]:
sample_pl = items_pl.sample(n=min(2000, items_pl.height), seed=1)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(sample_pl["price"].to_numpy(), sample_pl["freight_value"].to_numpy(), s=6, alpha=0.4)
ax.set_xscale("log")
ax.set_xlabel("price (log scale)")
ax.set_ylabel("freight value")
ax.set_title("Price vs freight (sample of 2,000 items)")
plt.tight_layout()
plt.show()

## 10. A taste of polars lazy mode

So far both libraries ran every step **immediately** (*eager* mode). Polars can also work **lazily**: `scan_csv` does not read the file yet. We describe the whole query first, and only `collect()` runs it. Because polars sees the **whole plan**, it can optimise it, for example by reading **only the columns it needs** from the file. On big files this saves time and memory.

The query below counts orders per status, like in 5.1.

In [ ]:
lazy_query = (
    pl.scan_csv(CSV["orders"])
    .group_by("order_status")
    .agg(pl.len().alias("orders"))
    .sort("orders", descending=True)
)

print(lazy_query.explain())   # the optimised plan (optional: just to look at it)
lazy_query.collect()

## 11. Cheat sheet: pandas versus polars

| Task | pandas | polars |
|---|---|---|
| Read a CSV | `pd.read_csv(path)` | `pl.read_csv(path)` |
| Shape / types | `df.shape`, `df.dtypes` | `df.shape`, `df.schema` |
| First rows | `df.head(3)` | `df.head(3)` |
| Select columns | `df[["a", "b"]]` | `df.select("a", "b")` |
| Filter rows | `df[df["a"] > 5]` | `df.filter(pl.col("a") > 5)` |
| New column | `df.assign(c=df["a"] * 2)` | `df.with_columns((pl.col("a") * 2).alias("c"))` |
| Sort | `df.sort_values("a")` | `df.sort("a")` |
| Count missing | `df.isna().sum()` | `df.null_count()` |
| Fill missing | `df["a"].fillna(0)` | `pl.col("a").fill_null(0)` |
| Count per value | `df["a"].value_counts()` | `df.group_by("a").agg(pl.len())` |
| Group and aggregate | `df.groupby("a")["b"].sum()` | `df.group_by("a").agg(pl.col("b").sum())` |
| Join | `df.merge(other, on="k", how="left")` | `df.join(other, on="k", how="left")` |
| Duplicates | `df.duplicated().sum()` | `df.is_duplicated().sum()` |
| Quantile | `df["a"].quantile(0.9)` | `pl.col("a").quantile(0.9, interpolation="linear")` |
| Lazy | not built in | `pl.scan_csv(path) ... .collect()` |

### Which one should I use?

- **pandas**: the standard. Best when you need the huge ecosystem (plotting, scikit-learn, most tutorials) or you work with small to medium data
- **polars**: great for larger data, clear and consistent code, and speed. On the Olist files (about 100,000 rows) both are fast, so the choice is mostly a matter of taste
- You can **convert** between them when needed: `pl.from_pandas(df)` and `df.to_pandas()` (the latter needs the `pyarrow` package)

## 13. Summary

### An EDA checklist you can reuse

1. **Size and shape**: rows, columns, a sample of rows
2. **Types**: dates are dates, identifiers are text, amounts are numbers
3. **Missing values**: how many, where, and do they make sense?
4. **Duplicates and keys**: is the key really unique?
5. **Categorical columns**: value counts; rare values; missing categories
6. **Numeric columns**: mean vs median, quantiles, skew, outliers, impossible values
7. **Time**: does the data cover the whole period? Are the first and last periods incomplete?
8. **Joins**: orphans, key uniqueness, and **row counts before and after**
9. **Relationships**: correlation, scatter plots

### Key ideas

| Idea | In one sentence |
|---|---|
| **EDA** | Looking at the data itself, before using it, to learn what is in it and what is wrong with it |
| **Missing is not always an error** | Always ask *why* it is missing (an undelivered order has no delivery date) |
| **Mean vs median** | A big gap means a skewed distribution with a few very large values |
| **Outlier** | An unusual value: look at it first, do not delete it automatically |
| **Join check** | After every join, compare the row counts and look for orphans |
| **Eager vs lazy** | Eager runs each step now; lazy plans the whole query first and then runs it |